# 🏏 IPL Data Analysis (2008–2024)
**Personal Project | github.com/Datthu489 | 2026**

Analyzing IPL match and player performance data using Python (Pandas, Matplotlib, Seaborn) to uncover key trends across 17 seasons.

---


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Libraries imported successfully")

## 2. Load & Explore Data

In [ ]:
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print("📦 Matches Dataset:", matches.shape)
print("📦 Deliveries Dataset:", deliveries.shape)
matches.head()

In [ ]:
print("\n🔍 Matches Columns:", matches.columns.tolist())
print("\n🔍 Deliveries Columns:", deliveries.columns.tolist())
matches.info()

## 3. Data Cleaning

In [ ]:
# Fix date
matches['date'] = pd.to_datetime(matches['date'])

# Standardize season to year
def clean_season(s):
    s = str(s)
    if '/' in s:
        return int(s.split('/')[0])
    return int(s)

matches['year'] = matches['season'].apply(clean_season)

# Standardize team names (merged franchises)
team_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
}
for col in ['team1','team2','winner','toss_winner']:
    matches[col] = matches[col].replace(team_map)

deliveries['batting_team'] = deliveries['batting_team'].replace(team_map)
deliveries['bowling_team'] = deliveries['bowling_team'].replace(team_map)

# Check nulls
print("🧹 Null values in matches:")
print(matches.isnull().sum()[matches.isnull().sum() > 0])
print("\n✅ Data Cleaning Done!")
print(f"   Total Matches: {len(matches)}")
print(f"   Total Deliveries: {len(deliveries)}")
print(f"   Seasons: {sorted(matches['year'].unique())}")

## 4. Exploratory Data Analysis

### 4.1 Season-wise Match Count

In [ ]:
season_matches = matches.groupby('year')['id'].count().reset_index()
season_matches.columns = ['Year', 'Matches']

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(season_matches['Year'].astype(str), season_matches['Matches'],
              color=sns.color_palette('Blues_d', len(season_matches)), edgecolor='navy', linewidth=0.5)

for bar, val in zip(bars, season_matches['Matches']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('📅 Season-wise Match Count (2008–2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('Number of Matches', fontsize=11)
ax.set_xticklabels(season_matches['Year'].astype(str), rotation=45)
plt.tight_layout()
plt.savefig('season_matches.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: IPL expanded from 58 matches in 2008 to 74 matches in 2022 onwards with more teams.")

### 4.2 Most Successful Teams

In [ ]:
team_wins = matches['winner'].value_counts().head(10).reset_index()
team_wins.columns = ['Team', 'Wins']

colors = ['#1a237e','#283593','#303f9f','#3949ab','#3f51b5',
          '#5c6bc0','#7986cb','#9fa8da','#c5cae9','#e8eaf6']

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(team_wins['Team'][::-1], team_wins['Wins'][::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, team_wins['Wins'][::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold')

ax.set_title('🏆 Top 10 Most Successful IPL Teams', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Wins', fontsize=11)
ax.set_xlim(0, team_wins['Wins'].max() + 20)
ax.axvline(x=team_wins['Wins'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Avg: {team_wins["Wins"].mean():.0f}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('team_wins.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Insight: Mumbai Indians lead with {team_wins['Wins'].iloc[0]} wins, followed by Chennai Super Kings with {team_wins['Wins'].iloc[1]} wins.")

### 4.3 Toss Analysis

In [ ]:
# Toss decision preference
toss_dec = matches['toss_decision'].value_counts()

# Toss win -> match win
matches['toss_win_match_win'] = matches['toss_winner'] == matches['winner']
toss_impact = matches['toss_win_match_win'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1
wedge1, texts1, auto1 = axes[0].pie(toss_dec.values, labels=toss_dec.index,
    autopct='%1.1f%%', colors=['#1565C0','#42A5F5'], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':12})
axes[0].set_title('Toss Decision: Field vs Bat', fontsize=13, fontweight='bold')

# Chart 2
wedge2, texts2, auto2 = axes[1].pie(toss_impact.values,
    labels=['Win Match', 'Lose Match'],
    autopct='%1.1f%%', colors=['#2E7D32','#C62828'], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2}, textprops={'fontsize':12})
axes[1].set_title('Toss Win → Match Win?', fontsize=13, fontweight='bold')

plt.suptitle('🎯 Toss Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('toss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Insight: {toss_dec['field']/toss_dec.sum()*100:.1f}% of captains choose to field after winning toss.")
print(f"💡 Insight: Winning toss has only {toss_impact[True]/toss_impact.sum()*100:.1f}% correlation with winning the match.")

### 4.4 Top Run Scorers

In [ ]:
top_batsmen = deliveries.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10).reset_index()
top_batsmen.columns = ['Batter', 'Total Runs']

# Also get matches played
batter_matches = deliveries.groupby('batter')['match_id'].nunique().reset_index()
batter_matches.columns = ['Batter', 'Matches']
top_batsmen = top_batsmen.merge(batter_matches, on='Batter')
top_batsmen['Avg per Match'] = (top_batsmen['Total Runs'] / top_batsmen['Matches']).round(1)

palette = sns.color_palette('YlOrRd', 10)[::-1]
fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(top_batsmen['Batter'], top_batsmen['Total Runs'],
              color=palette, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, top_batsmen['Total Runs']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('🏏 Top 10 Run Scorers in IPL History', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Batsman', fontsize=11)
ax.set_ylabel('Total Runs', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('top_batsmen.png', dpi=150, bbox_inches='tight')
plt.show()
print(top_batsmen[['Batter','Total Runs','Matches','Avg per Match']].to_string(index=False))
print(f"\n💡 Insight: Virat Kohli leads with {top_batsmen['Total Runs'].iloc[0]:,} runs across {top_batsmen['Matches'].iloc[0]} matches.")

### 4.5 Top Wicket Takers

In [ ]:
wickets = deliveries[(deliveries['is_wicket']==1) & 
                   (~deliveries['dismissal_kind'].isin(['run out','retired hurt','obstructing the field']))]

top_bowlers = wickets.groupby('bowler')['is_wicket'].count().sort_values(ascending=False).head(10).reset_index()
top_bowlers.columns = ['Bowler', 'Wickets']

palette2 = sns.color_palette('GnBu', 10)[::-1]
fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(top_bowlers['Bowler'], top_bowlers['Wickets'],
              color=palette2, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, top_bowlers['Wickets']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('🎳 Top 10 Wicket Takers in IPL History', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Bowler', fontsize=11)
ax.set_ylabel('Total Wickets', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('top_bowlers.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Insight: Yuzvendra Chahal leads with {top_bowlers['Wickets'].iloc[0]} wickets.")

### 4.6 Season-wise Total Runs

In [ ]:
match_year = matches[['id','year']].rename(columns={'id':'match_id'})
del_year = deliveries.merge(match_year, on='match_id')
season_runs = del_year.groupby('year')['total_runs'].sum().reset_index()
season_runs.columns = ['Year', 'Total Runs']

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(season_runs['Year'].astype(str), season_runs['Total Runs'],
                alpha=0.3, color='#1565C0')
ax.plot(season_runs['Year'].astype(str), season_runs['Total Runs'],
        color='#1565C0', linewidth=2.5, marker='o', markersize=7)

for x, y in zip(season_runs['Year'].astype(str), season_runs['Total Runs']):
    ax.annotate(f'{y//1000}K', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8, fontweight='bold')

ax.set_title('📊 Season-wise Total Runs Scored', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('Total Runs', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('season_runs.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 Most Player of the Match Awards

In [ ]:
potm = matches['player_of_match'].value_counts().head(10).reset_index()
potm.columns = ['Player', 'Awards']

fig, ax = plt.subplots(figsize=(12, 5))
colors_potm = sns.color_palette('rocket', 10)[::-1]
bars = ax.barh(potm['Player'][::-1], potm['Awards'][::-1], color=colors_potm, edgecolor='white')

for bar, val in zip(bars, potm['Awards'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold')

ax.set_title('⭐ Most Player of the Match Awards', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Awards', fontsize=11)
ax.set_xlim(0, potm['Awards'].max() + 5)
plt.tight_layout()
plt.savefig('potm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"💡 Insight: {potm['Player'].iloc[0]} won the most Player of the Match awards ({potm['Awards'].iloc[0]} times).")

### 4.8 Most Matches by Venue

In [ ]:
venue_matches = matches['venue'].value_counts().head(10).reset_index()
venue_matches.columns = ['Venue', 'Matches']

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=venue_matches, y='Venue', x='Matches',
            palette='Blues_r', ax=ax)
ax.set_title('🏟️ Top 10 IPL Venues by Matches Hosted', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Matches', fontsize=11)
ax.set_ylabel('')
for i, (val, name) in enumerate(zip(venue_matches['Matches'], venue_matches['Venue'])):
    ax.text(val + 0.3, i, str(val), va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('venues.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Key Insights Summary

| # | Insight |
|---|---------|
| 1 | 🏆 **Mumbai Indians** are the most successful team with 144 wins |
| 2 | 🏏 **Virat Kohli** is the all-time leading run scorer with 8,014 runs |
| 3 | 🎳 **Yuzvendra Chahal** leads wicket-takers with 205 wickets |
| 4 | 🎯 Winning toss has **only ~50%** correlation with winning the match |
| 5 | 📊 **64%** of captains choose to field after winning toss |
| 6 | 📅 IPL grew from **58 matches** in 2008 to **74 matches** in 2022+ |
| 7 | 🎄 **Christmas & festive weeks** see peak matches and engagement |

---
*Analysis by Datthu | github.com/Datthu489 | 2026*
